# 1. Initial labels from rules

Give a label only to the frames where the tracking makes it clear:
- **H / A**: a player clearly has the ball at their feet;
- **D**: play is clearly stopped;
- **carry**: between two clear moments of the same team, that team keeps the ball.

Everything else stays unlabelled (`-`). The code lives in `possession/`; this notebook shows each
step with a small example first, then applies it to the whole match.

In [ ]:
import numpy as np
import pandas as pd

from possession import rules
from possession.data import (ATHLETIC, BARCA, DEAD, FPS, GOALKEEPER_SLOT, HALVES, LABEL_NAME, SECOND_HALF_KICKOFF,
                             TEAM_NAME, UNLABELLED, attacking_direction, ball_distance_to_line, ball_speed,
                             distance_from_own_goal, graded_frames, load_events, load_tracking)


def show(labels):
    """[0, 0, 2, -1, 1] -> 'HHD-A'"""
    return ''.join(LABEL_NAME[x] for x in labels)

## 1. The data

One row per frame (25 per second). Each frame has 46 player **slots** (tracker IDs, each fixed
to one team), the ball, and the Smart Tagging intervals. Positions are in metres.

In [ ]:
tracking = load_tracking()
events = load_events()
players_xy, players_speed = tracking['players_xy'], tracking['players_speed']
ball_xy, slot_team = tracking['ball_xy'], tracking['slot_team']
print('players_xy', players_xy.shape, '| ball_xy', ball_xy.shape, '|', len(events), 'tags')

Example: one frame. Only ~20 of the 46 slots are in use at a time.

In [ ]:
FRAME = 30683  # a moment where a Barça player has the ball
in_use = ~np.isnan(players_xy[FRAME, :, 0])
pd.DataFrame({'slot': np.flatnonzero(in_use) + 1,
              'team': [TEAM_NAME[t] for t in slot_team[in_use]],
              'x': players_xy[FRAME, in_use, 0], 'y': players_xy[FRAME, in_use, 1],
              'speed': players_speed[FRAME, in_use]}).astype({'x': float, 'y': float, 'speed': float}).round(1).head(6)

In [ ]:
print('ball at frame', FRAME, ':', ball_xy[FRAME].round(1))

## 2. Graded frames

Only the two halves are scored; the 2nd half kicks off at frame 113,742 (FAQ.md).

In [ ]:
graded = graded_frames()
print(f'{graded.sum()} graded frames, halves: {HALVES}')

## 3. Ball signals

- `visible`: the ball has a position on this frame;
- `speed`: m/s, from how far the ball moves over 4 frames (the file's ball speed is unusable, explore_tracking.ipynb section 3);
- `to_line`: distance to the nearest line, negative = outside the pitch.

In [ ]:
visible = ~np.isnan(ball_xy[:, 0])
speed = ball_speed(ball_xy)
to_line = ball_distance_to_line(ball_xy)
print(f'ball visible on {visible[graded].mean():.1%} of graded frames')
pd.DataFrame({'x': ball_xy[FRAME:FRAME + 5, 0], 'y': ball_xy[FRAME:FRAME + 5, 1],
              'speed': speed[FRAME:FRAME + 5], 'to_line': to_line[FRAME:FRAME + 5]},
             index=range(FRAME, FRAME + 5)).round(2)

## 4. Attacking direction

A team attacks away from its own goalkeeper, and the sides switch at halftime.
`from_own_goal` = each player's distance from their own goal line (0 = own goal, 105 = opponent's goal).

In [ ]:
direction = attacking_direction(players_xy)
from_own_goal = distance_from_own_goal(players_xy, slot_team, direction)
pd.DataFrame([{'half': h, 'team': TEAM_NAME[t], 'attacks towards': 'x = 105' if d > 0 else 'x = 0'}
              for (t, h), d in direction.items()])

In [ ]:
for team in (BARCA, ATHLETIC):
    slot = np.flatnonzero(in_use & (slot_team == team))[0]
    print(f'frame {FRAME}, slot {slot + 1} ({TEAM_NAME[team]}): '
          f'x = {players_xy[FRAME, slot, 0]:.1f} m -> {from_own_goal[FRAME, slot]:.1f} m from own goal')

## 5. H / A: clear control

A team has the ball **clearly** when, for at least 5 frames in a row:
the ball is inside the pitch and slower than 6 m/s, a player is within 1.5 m of it, and the nearest
opponent is at least 1.5 m further away. (Numbers from rules_analysis.ipynb section 2.)

First the distance from the ball to the nearest player of each team:

In [ ]:
dist_barca, dist_athletic = rules.nearest_player_distance(players_xy, slot_team, ball_xy)
pd.DataFrame({'nearest Barça (m)': dist_barca[FRAME:FRAME + 5], 'nearest Athletic (m)': dist_athletic[FRAME:FRAME + 5],
              'ball speed': speed[FRAME:FRAME + 5]}, index=range(FRAME, FRAME + 5)).round(2)

Toy example, 10 frames: Barça close for 6 frames (enough), then Athletic for 3 (too short).

In [ ]:
toy_barca = np.array([0.5, 0.6, 0.5, 0.7, 0.8, 0.6, 5.0, 5.0, 5.0, 5.0])
toy_athletic = np.array([4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 0.5, 0.6, 0.5, 4.0])
toy_on = np.ones(10, bool)
toy = rules.clear_control(toy_barca, toy_athletic, np.full(10, 2.0), np.full(10, 20.0), toy_on)
print(show(toy))

In [ ]:
control = rules.clear_control(dist_barca, dist_athletic, speed, to_line, graded)
print(f'clear control on {np.mean(control[graded] != UNLABELLED):.1%} of graded frames: '
      f'H {np.sum(control == BARCA)}, A {np.sum(control == ATHLETIC)}')

## 6. Dead ball

Each rule gives stretches `(start, end)` of stopped play. Most end at the **restart**: the first
moment the ball is inside the pitch and moving faster than 3 m/s for 10 frames (rules_analysis.ipynb section 3).

Toy example: ball still, then moving for 3 frames (too short), then moving for 10 → restart at frame 8.

In [ ]:
toy_speed = np.array([0, 0, 0, 5, 5, 5, 0, 0] + [5] * 10 + [0, 0], float)
print('restart found at frame', rules.restart_finder(toy_speed, np.full(20, 10.0))(0))
restart = rules.restart_finder(speed, to_line)

### R1 ball out
The ball is seen more than 0.5 m outside the pitch (rules_analysis.ipynb section 3).

In [ ]:
r1 = rules.ball_out(to_line, graded, restart)
print(len(r1), 'stretches, e.g.', r1[:3])

### R2 ball lost at a line
The ball disappears within 2 m of a line (rules_analysis.ipynb section 3: such balls come back at
a line 48% of the time vs 18% for other gaps, so this is likely, not certain).

In [ ]:
r2 = rules.ball_lost_at_line(to_line, graded, restart)
print(len(r2), 'stretches')

### R3 ball still
The ball does not move (< 0.5 m/s) for 1 s (rules_analysis.ipynb section 3), **except** when a
goalkeeper stands within 1.5 m of it outside the 6-yard box: that is the keeper holding it, live play.

In [ ]:
held = rules.keeper_holding(players_xy, ball_xy, GOALKEEPER_SLOT.values())
r3 = rules.ball_still(speed, held, graded, restart)
r3_all = rules.ball_still(speed, np.zeros_like(held), graded, restart)
print(f'{len(r3_all)} still-ball stretches, {len(r3_all) - len(r3)} dropped as keeper holding -> {len(r3)}')

### R4 set-piece tag
Dead from 8 s before each `SET PIECES` tag until the restart (rules_analysis.ipynb section 3).

In [ ]:
r4 = rules.set_piece_tag(events, graded, restart)
print(len(r4), 'stretches')

### R5 players walking
Most players walking (< 2 m/s) is typical of stopped play. Check: how often are ≥ 90% walking, on
frames we already know are live (clear control), known dead (2nd half before kickoff), and inside R1–R4?

In [ ]:
walking = rules.share_walking(players_speed)
before_kickoff = np.zeros(len(graded), bool)
before_kickoff[HALVES[1][1]:SECOND_HALF_KICKOFF] = True
_, dead_r1_r4 = rules.merge_stoppages({'R1': r1, 'R2': r2, 'R3': r3, 'R4': r4}, graded)
for name, frames in [('clear control (live)', (control != UNLABELLED) & ~dead_r1_r4),
                     ('2nd half before kickoff (dead)', before_kickoff),
                     ('inside R1-R4 (dead)', dead_r1_r4 & ~before_kickoff)]:
    print(f'{name:<32} >= 90% walking on {np.mean(walking[frames] >= 0.9):.0%} of frames')

Rare in live play, common in dead play. To stay safe, R5 also needs the **ball missing** and the
walking to last **10 s**.

In [ ]:
r5 = rules.players_walking(walking, visible, graded)
print(len(r5), 'stretches, length (s):', np.percentile([(b - a) / FPS for a, b in r5], [10, 50, 90]).round(1))

### R6 kickoff after a goal
Before a kickoff, ≥ 90% of each team stands in its own half while most players walk. The ball is
usually missing or being carried back, so only the players are used. Dead from the first such
moment in the 150 s before a `Kick Off (After Goal)` tag until the restart.

In [ ]:
own_half = {t: rules.share_in_own_half(from_own_goal, slot_team, t) for t in (BARCA, ATHLETIC)}
in_formation = graded & (walking > 0.5) & (own_half[BARCA] >= 0.9) & (own_half[ATHLETIC] >= 0.9)
r6 = rules.kickoff_after_goal(events, in_formation, restart)
n_kickoffs = sum(ev['tags'].get('Type') == 'Kick Off (After Goal)' for ev in events)
print(f'{n_kickoffs} kickoffs after a goal, formation found before {len(r6)}; dead for (s):',
      [round((b - a) / FPS) for a, b in r6])

### Merging into stoppages
Stretches closer than 2 s become one stoppage. Toy example (gap 2 s = 50 frames):

In [ ]:
toy_stoppages, _ = rules.merge_stoppages({'R1': [(0, 10)], 'R2': [(100, 160)], 'R4': [(190, 250)]}, np.ones(300, bool))
toy_stoppages

In [ ]:
spans = {'R1 ball out': r1, 'R2 lost at a line': r2, 'R3 ball still': r3, 'R4 set-piece tag': r4,
         'R5 players walking': r5, 'R6 kickoff after goal': r6,
         '2nd half before kickoff': rules.second_half_pre_kickoff()}
stoppages, dead = rules.merge_stoppages(spans, graded)
print(f'{len(stoppages)} stoppages, dead on {dead[graded].mean():.1%} of graded frames')

How much each rule adds: the dead share with that one rule switched off.

In [ ]:
pd.DataFrame([{'without': name, 'dead share': f'{rules.merge_stoppages({k: v for k, v in spans.items() if k != name}, graded)[1][graded].mean():.1%}'}
              for name in spans])

### Which D to trust for training
R2 (lost at a line, ~50% right) and R5 (walking) are the uncertain rules. Their frames stay D in the
labels, but the model in notebook 2 won't learn D from frames that **only** they mark dead, so it
doesn't learn "ball missing = dead".

In [ ]:
_, trusted_dead = rules.merge_stoppages({k: v for k, v in spans.items() if k not in rules.UNCERTAIN_DEAD_RULES}, graded)
print(f'dead {dead[graded].mean():.1%} of graded: trusted {trusted_dead[graded].mean():.1%}, '
      f'only R2/R5 {(dead & ~trusted_dead)[graded].mean():.1%}')

## 7. Carry and the final labels

1. **D wins**: a dead frame is D even if a player is close to the ball.
2. **Carry**: between two clear-control frames the earlier team keeps the ball, unless there is a
   D in between or the ball is missing too long: more than **1 s** when the teams on the two sides
   differ (we can't see where the switch happened), more than **10 s** when it is the same team
   on both sides.

Why 10 s: ball-missing stretches between two clear moments are almost never 1–5 s long, so a 5 s
limit adds nothing; 10 s (the same length R5 uses) gives the model ~2,400 live frames with the ball
missing to learn from, instead of 23.

Toy examples (`.` = no clear control, `D` = dead, `m` = ball missing, 25 frames = 1 s):

In [ ]:
def toy_carry(pattern):
    code = {'H': BARCA, 'A': ATHLETIC}
    control = np.array([code.get(c, UNLABELLED) for c in pattern])
    dead = np.array([c == 'D' for c in pattern])
    visible = np.array([c != 'm' for c in pattern])
    return show(rules.carry(control, dead, visible, np.ones(len(pattern), bool)))


for name, pattern in [('pause', 'HH..HH'), ('switch', 'HH..AA'), ('stoppage between', 'HH.D.AA'),
                      ('ball missing 2 s, same team', 'HH' + 'm' * 50 + 'HH'),
                      ('ball missing 2 s, switch', 'HH' + 'm' * 50 + 'AA'),
                      ('ball missing 12 s, same team', 'HH' + 'm' * 300 + 'HH')]:
    out = toy_carry(pattern)
    print(f'{name:<30} {pattern[:8]}... -> {out[:8]}...  (filled: {out.count("H") + out.count("A") - pattern.count("H") - pattern.count("A")} frames)')

In [ ]:
labels = rules.carry(control, dead, visible, graded)
counts = pd.Series({'H': np.sum(labels[graded] == BARCA), 'A': np.sum(labels[graded] == ATHLETIC),
                    'D': np.sum(labels[graded] == DEAD),
                    '- ball visible': np.sum((labels == UNLABELLED) & graded & visible),
                    '- ball missing': np.sum((labels == UNLABELLED) & graded & ~visible)})
pd.DataFrame({'frames': counts, '% of graded': (100 * counts / graded.sum()).round(1)})

## 8. Check
`rules.initial_labels()` runs this whole notebook in one call (notebook 2 uses it). Same result?

In [ ]:
labels_again, trusted_dead = rules.initial_labels(tracking, events)
assert (labels_again == labels).all()
print('same labels')

## Summary

| Label | % of graded frames | From |
|---|---|---|
| H | 26.4 | clear control 9.9 + carry |
| A | 16.0 | clear control 6.1 + carry |
| D | 39.5 | rules R1–R6 + 2nd half before kickoff (R1–R4 alone: 27.9); 27.3 from trusted rules, 12.2 only from R2/R5 |
| unlabelled | 18.0 | 9.3 with the ball visible, 8.7 with it missing |

- **Biggest D rules:** R5 players walking (dead share drops to 30.5% without it), then R4 set-piece tags (35.3%).
- **Numbers not from earlier analysis** (chosen here): keeper within 1.5 m (R3 exception), R5's
  90% / 2 m/s / 10 s (checked in the R5 cell), R6's 90% / 2 s / 150 s, carry limits 1 s / 10 s
  (10 s explained in section 7), merge gap 2 s.